# Session ID Reuse — What's Wrong and How Much It Matters

## What this notebook answers

Every reactivation metric used in `apec_analysis.ipynb` and
`apec_analysis_v2.ipynb` attributes a marketing send's reactivation credit
through a chain: **send &rarr; matched session (by UTM, within a time
window) &rarr; a demand event that shares that session's `active_session_id`
in `curated.client_reactivation_demand_events`.**

That last join step -- matching purely on `active_session_id` -- assumes a
session id uniquely identifies one browsing session at one point in time.
**It doesn't always.** This notebook shows, with real query results:

1. How often a demand event's `created_ts` (when it was recorded) drifts far
   from `session_datetime_in_utc` (when the session it's linked to actually
   happened) -- and why.
2. How often the same `active_session_id` shows up on more than one demand
   event, sometimes for a different client entirely.
3. How much this actually distorts the reactivation rate the other
   notebooks report, broken out **by client lifecycle state** (Active,
   Lapsed, Dormant, Dormant 3+ yrs) -- since the answer turns out to differ
   sharply by state.

For concrete example rows of the patterns described here, see the
companion notebook `session_id_sample.ipynb` in this same directory.

## Bottom line

Comparing the naive (current, production-matching) join against a
corrected join that also requires the demand event to have been created
close to the send:

| Client state | Naive rate | Corrected rate | Overstatement |
|---|---|---|---|
| Active | 0.1275% | 0.0976% | **+30.7% (1.31x)** |
| Lapsed | 3.0854% | 3.0831% | +0.07% |
| Dormant | 2.1335% | 2.1326% | +0.04% |
| Dormant 3+ yrs | 1.3100% | 1.3094% | +0.05% |

The distortion is concentrated almost entirely in the **Active** state and
is negligible everywhere else -- which is one of the reasons
`apec_analysis.ipynb` / `apec_analysis_v2.ipynb` and their presentation
decks exclude Active clients from every reactivation-rate comparison.

In [1]:
import time

import pandas as pd
from amphibian import get_data_accessor

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

def query(sql, max_attempts=4, retry_delay_seconds=20):
    # The longer queries in this project occasionally hit a transient connection
    # drop (network/VPN blip, not a query bug -- the same query succeeds on retry).
    for attempt in range(1, max_attempts + 1):
        try:
            return get_data_accessor(engines=['presto']).fetch_sql(sql=sql, error_on_empty=False)
        except Exception as e:
            if attempt == max_attempts:
                raise
            print(f"query() attempt {attempt}/{max_attempts} failed ({type(e).__name__}: {e}); retrying in {retry_delay_seconds}s...")
            time.sleep(retry_delay_seconds)

START_DATE = '2025-01-01'
END_DATE = '2026-05-31'
SESSION_ATTRIBUTION_WINDOW_DAYS = 7
ACTIVE_MAX_DAYS = 120
LAPSED_MAX_DAYS = 365
DORMANT_3YR_MIN_DAYS = 1095

## Step 1 — How the current reactivation join works

Every reactivation query in this project's other notebooks follows the same
three-step pattern:

1. **`matched_sessions`** -- join each eligible send to a real browsing
   session in `curated.user_session_conversion_metrics`, matched on
   `client_id` + `utm_content`, where the session happened within
   `SESSION_ATTRIBUTION_WINDOW_DAYS` (7 days) after the send.
2. **`attributed_demand`** -- join that session's `active_session_id` to
   `curated.client_reactivation_demand_events.active_session_id`, filtering
   to `demand_type IN ('fix', 'direct_buy')`.
3. Count a client as reactivated if any of their sends produced an
   attributed demand event.

Step 2 is a pure equi-join on `active_session_id`, with **no constraint
relating the demand event's own timing to the send or session it's being
credited to**. `curated.client_reactivation_demand_events` happens to carry
two independent timestamps on every row -- `session_datetime_in_utc` (when
the linked session happened) and `created_ts` (when this demand row itself
was recorded) -- so whether those two ever drift apart is directly
checkable, which is what Step 2 below does.

## Step 2 — Does `created_ts` ever drift from `session_datetime_in_utc`?

If `active_session_id` cleanly identified one specific browsing session,
`created_ts` should sit within minutes to hours of `session_datetime_in_utc`
for every demand event tied to it. This checks that gap, split by
`demand_type` and by whether the `fix` was generated by an autoship
subscription (`fix_demand_autoship_subscription_id IS NOT NULL`).

In [2]:
gap_query = f"""--sql
SELECT
    demand_type,
    fix_demand_autoship_subscription_id IS NOT NULL AS is_autoship,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT active_session_id) AS n_distinct_session_ids,
    approx_percentile(date_diff('day', session_datetime_in_utc, created_ts), 0.5) AS p50_gap_days,
    approx_percentile(date_diff('day', session_datetime_in_utc, created_ts), 0.9) AS p90_gap_days,
    MAX(date_diff('day', session_datetime_in_utc, created_ts)) AS max_gap_days,
    SUM(CASE WHEN date_diff('day', session_datetime_in_utc, created_ts) > 30 THEN 1 ELSE 0 END) AS n_gap_over_30d
FROM curated.client_reactivation_demand_events
WHERE demand_type IN ('fix', 'direct_buy')
  AND created_ts >= DATE '{START_DATE}'
GROUP BY 1, 2
ORDER BY 1, 2
"""

gap_df = query(gap_query)
gap_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,demand_type,is_autoship,n_rows,n_distinct_session_ids,p50_gap_days,p90_gap_days,max_gap_days,n_gap_over_30d
0,direct_buy,False,431839,418201,0,0,1,0
1,fix,False,1052305,952462,0,0,1,0
2,fix,True,3790059,441005,0,0,2830,133050


**Non-autoship demand events are clean.** `direct_buy` and non-autoship
`fix` events show a maximum gap of just 1 day between the session and when
the demand row was created -- exactly what a tightly-scoped session id
should look like.

**Autoship-driven `fix` events are the problem.** Once a subscription is
created from an original session, every subsequent shipment generates a new
`fix` demand row -- for as long as the subscription stays active -- and
every one of those rows still carries the **original** `active_session_id`
from signup, not a new session tied to that later shipment. The gap
between that original session and a shipment created much later reaches
into the thousands of days, and tens of thousands of rows sit more than 30
days apart.

## Step 3 — Is `active_session_id` itself reused across multiple demand events?

Given Step 2's finding, the next question is how often one `active_session_id`
ends up attached to more than one demand row -- and whether it's ever
shared across genuinely different clients (a stronger, harder-to-defend
form of reuse than one client's own subscription re-firing).

In [3]:
reuse_query = f"""--sql
WITH per_sid AS (
    SELECT
        active_session_id,
        COUNT(*) AS n_rows,
        COUNT(DISTINCT client_id) AS n_distinct_clients,
        date_diff('day', MIN(created_ts), MAX(created_ts)) AS created_span_days
    FROM curated.client_reactivation_demand_events
    WHERE demand_type IN ('fix', 'direct_buy')
      AND created_ts >= DATE '{START_DATE}'
      AND active_session_id IS NOT NULL
    GROUP BY 1
)
SELECT
    COUNT(*) AS total_nonnull_session_ids,
    SUM(CASE WHEN n_rows > 1 THEN 1 ELSE 0 END) AS session_ids_reused_across_rows,
    SUM(CASE WHEN n_distinct_clients > 1 THEN 1 ELSE 0 END) AS session_ids_shared_across_clients,
    approx_percentile(created_span_days, 0.5) FILTER (WHERE n_rows > 1) AS p50_span_days_when_reused,
    approx_percentile(created_span_days, 0.95) FILTER (WHERE n_rows > 1) AS p95_span_days_when_reused,
    MAX(created_span_days) AS max_span_days
FROM per_sid
"""

reuse_df = query(reuse_query)
reuse_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,total_nonnull_session_ids,session_ids_reused_across_rows,session_ids_shared_across_clients,p50_span_days_when_reused,p95_span_days_when_reused,max_span_days
0,1401896,503522,16780,0,76,594


Roughly **36% of distinct, non-null session ids** attached to a demand
event show up on more than one demand row, and about **1.2% are shared
across genuinely different clients** -- a session id, by definition, should
never belong to more than one person. Concrete example rows for both
patterns (same-client long-gap reuse, and cross-client sharing) are in
`session_id_sample.ipynb`.

Two distinct failure modes are at play:

- **Same-client, long-gap reuse (the dominant pattern).** An autoship
  subscription keeps generating `fix` demand rows under its original
  session id for as long as it stays active -- sometimes over 500 days
  later.
- **Cross-client collisions (rarer, ~1.2% of session ids).** The same
  session id appears against different clients, almost always clustered
  within minutes of each other -- consistent with an id-generation
  collision rather than a real reuse of one person's session, and too rare
  on its own to meaningfully affect the rates below.

## Step 4 — How much does this inflate the reactivation rate, by client state?

The current join credits a send with reactivation if its matched session's
`active_session_id` appears **anywhere** in
`curated.client_reactivation_demand_events`, regardless of when that demand
row was created (**naive**, matches every other notebook in this project).

A straightforward fix: additionally require the demand event's `created_ts`
to fall within the **same attribution window** used to match the session to
the send in the first place (`SESSION_ATTRIBUTION_WINDOW_DAYS`, 7 days) --
i.e., the reactivation must have actually happened close to the send, not
merely share a session id with something that happened far later
(**corrected**).

This reruns the full attribution pipeline both ways, split by client
lifecycle state as of each send (four states, including Active -- this
notebook's whole point is to explain the size of the Active distortion, so
Active is deliberately included here even though the other notebooks
exclude it).

In [4]:
STATES = ["Active", "Lapsed", "Dormant", "Dormant 3+ yrs"]

def impact_query_for_state(state):
    return f"""--sql
WITH sends AS (
    SELECT client_id, sent_timestamp, send_utm_content
    FROM blueshift.campaign_activity_kpis
    WHERE execution_date >= DATE \'{START_DATE}\'
      AND execution_date <= DATE \'{END_DATE}\'
      AND holdout_group = 0
),
state_at_send AS (
    SELECT
        s.client_id, s.sent_timestamp, s.send_utm_content,
        CASE
            WHEN j.client_state_detail = \'Never Active\' THEN \'Never Active\'
            WHEN date_diff(\'day\', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {ACTIVE_MAX_DAYS} THEN \'Active\'
            WHEN date_diff(\'day\', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {LAPSED_MAX_DAYS} THEN \'Lapsed\'
            WHEN date_diff(\'day\', date(j.last_buyable_checkout_ts), date(s.sent_timestamp)) <= {DORMANT_3YR_MIN_DAYS} THEN \'Dormant\'
            ELSE \'Dormant 3+ yrs\'
        END AS lifecycle_state
    FROM sends s
    INNER JOIN curated.checkout_based_client_state_journal j
        ON s.client_id = j.client_id
        AND s.sent_timestamp >= j.start_timestamp
        AND s.sent_timestamp <  j.end_timestamp
),
eligible_sends AS (
    SELECT * FROM state_at_send WHERE lifecycle_state = \'{state}\'
),
matched_sessions AS (
    SELECT e.client_id, e.sent_timestamp, u.active_session_id
    FROM eligible_sends e
    INNER JOIN curated.user_session_conversion_metrics u
        ON u.client_id = e.client_id
        AND u.utm_source = \'blueshift\'
        AND u.utm_content = e.send_utm_content
        AND u.datetime_in_utc >= e.sent_timestamp
        AND u.datetime_in_utc <  e.sent_timestamp + INTERVAL \'{SESSION_ATTRIBUTION_WINDOW_DAYS}\' DAY
    WHERE u.date_in_utc >= DATE \'{START_DATE}\'
),
naive_attributed AS (
    -- current production join logic: matches on active_session_id alone
    SELECT DISTINCT ms.client_id, ms.sent_timestamp
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events de
        ON de.active_session_id = ms.active_session_id
    WHERE de.demand_type IN (\'fix\', \'direct_buy\')
),
corrected_attributed AS (
    -- same join, but the demand event must also have been CREATED within the
    -- same attribution window as the session match, so a demand event created
    -- months/years later (e.g. a recurring autoship shipment) can\'t get
    -- credited to a send just because it shares a long-lived session id
    SELECT DISTINCT ms.client_id, ms.sent_timestamp
    FROM matched_sessions ms
    INNER JOIN curated.client_reactivation_demand_events de
        ON de.active_session_id = ms.active_session_id
    WHERE de.demand_type IN (\'fix\', \'direct_buy\')
      AND de.created_ts >= ms.sent_timestamp
      AND de.created_ts <  ms.sent_timestamp + INTERVAL \'{SESSION_ATTRIBUTION_WINDOW_DAYS}\' DAY
),
client_send_level AS (
    SELECT
        e.client_id, e.sent_timestamp,
        MAX(CASE WHEN n.client_id IS NOT NULL THEN 1 ELSE 0 END) AS naive_reactivated,
        MAX(CASE WHEN c.client_id IS NOT NULL THEN 1 ELSE 0 END) AS corrected_reactivated
    FROM eligible_sends e
    LEFT JOIN naive_attributed n
        ON e.client_id = n.client_id AND e.sent_timestamp = n.sent_timestamp
    LEFT JOIN corrected_attributed c
        ON e.client_id = c.client_id AND e.sent_timestamp = c.sent_timestamp
    GROUP BY 1, 2
),
client_level AS (
    SELECT
        client_id,
        MAX(naive_reactivated) AS naive_reactivated,
        MAX(corrected_reactivated) AS corrected_reactivated
    FROM client_send_level
    GROUP BY 1
)
SELECT
    \'{state}\' AS lifecycle_state,
    COUNT(DISTINCT client_id) AS unique_clients_sent,
    SUM(naive_reactivated) AS naive_reactivated_clients,
    SUM(corrected_reactivated) AS corrected_reactivated_clients,
    CAST(SUM(naive_reactivated) AS DOUBLE) / COUNT(DISTINCT client_id) AS naive_reactivation_rate,
    CAST(SUM(corrected_reactivated) AS DOUBLE) / COUNT(DISTINCT client_id) AS corrected_reactivation_rate
FROM client_level
"""

# Run one state at a time -- each query is far lighter than one combined
# 4-state query, and this project\'s Presto cluster has shown transient
# worker-node instability (REMOTE_TASK_ERROR / RejectedExecutionException)
# on heavier combined joins; splitting by state keeps each one small enough
# to finish reliably even when a worker drops mid-query.
impact_results = []
for state in STATES:
    print(f"Running impact query for state={state}...")
    impact_results.append(query(impact_query_for_state(state)))

impact_df = pd.concat(impact_results, ignore_index=True)
impact_df["overstatement_pct"] = (impact_df["naive_reactivation_rate"] / impact_df["corrected_reactivation_rate"] - 1) * 100
impact_df

Running impact query for state=Active...


/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Running impact query for state=Lapsed...


/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Running impact query for state=Dormant...


/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Running impact query for state=Dormant 3+ yrs...


/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,lifecycle_state,unique_clients_sent,naive_reactivated_clients,corrected_reactivated_clients,naive_reactivation_rate,corrected_reactivation_rate,overstatement_pct
0,Active,3023509,3855,2950,0.001275,0.000976,30.677966
1,Lapsed,2427564,74901,74845,0.030854,0.030831,0.074821
2,Dormant,2788578,59493,59468,0.021335,0.021326,0.042039
3,Dormant 3+ yrs,4514343,59139,59109,0.013100,0.013094,0.050754


## Step 5 — Interpretation

The overstatement in the naive (current, production-matching) join versus
the corrected one is concentrated almost entirely in **Active** -- the state
where it was flagged the reactivation rate looked implausibly low,
and the state this project's other notebooks exclude for the separate,
more fundamental reason that `curated.client_reactivation_demand_events`
has zero rows for clients who were ever truly reactivating *from* Active
(Active clients aren't "returning" in the first place). Lapsed, Dormant,
and Dormant 3+ yrs move by a fraction of a percentage point -- confirming
the findings already published in `apec_analysis.ipynb`,
`apec_analysis_v2.ipynb`, and their presentation decks are not affected by
this leak.

## Limitations to keep in mind

- **The "corrected" 7-day window is one reasonable definition of "properly
  attributed," not the only possible one.** A demand event created just
  outside that window could still be a legitimate, if slightly delayed,
  response to the send -- so this correction likely overcorrects a little
  at the margin, not just undercorrects.
- **This notebook is diagnostic, not a rerun of the published analysis.**
  It doesn't change any number already published in `apec_analysis.ipynb`,
  `apec_analysis_v2.ipynb`, or their decks -- those already exclude Active
  for a more fundamental data-availability reason, and Step 4 above
  confirms the leak barely touches the three states those notebooks do
  report on.
- **`created_ts` reflects when Stitch Fix's systems recorded the demand
  event, not a perfectly clean signal of client intent timing** -- ordinary
  processing delays could shift it by minutes to hours. The multi-month,
  sometimes multi-year gaps found in Step 2 are far beyond anything
  explainable by that kind of lag.